# Convolutional Neural Networks — Practice Notebook
### Based on VE445 Lecture 7 (Shanghai Jiao Tong University)

This notebook follows the lecture slides section by section. Every cell is runnable.  
Cells marked `# YOUR CODE HERE` are for you to complete.

**Sections:**
1. Motivation — why not just use fully-connected?
2. Convolution from scratch (NumPy)
3. Sparse connections and weight sharing
4. Stride and padding
5. Pooling
6. Activation functions (ReLU)
7. Dropout and weight initialisation
8. Batch normalisation
9. Build and train a full CNN (PyTorch)
10. Famous architectures — AlexNet, VGGNet, ResNet

**Requirements:** `pip install numpy matplotlib torch torchvision`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torchvision import datasets, transforms
    HAVE_TORCH = True
    torch.manual_seed(42)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
except ImportError:
    HAVE_TORCH = False
    print("PyTorch not installed — sections 1-8 still run. pip install torch torchvision")


---
## Section 1 — Motivation: why not fully connected?

A 32×32 RGB image has 32×32×3 = **3,072 pixels**.  
A fully connected layer connecting that to 1000 hidden units = **3,072 × 1000 = 3.07 million parameters** — just for the first layer.  
For 224×224 images (standard ImageNet size): **150,528 × 1000 = 150 million parameters** in one layer.

This is wasteful because:
- Nearby pixels are correlated — the network should exploit spatial structure
- The same pattern (an edge) can appear anywhere — weights should be shared


In [ ]:
# Show parameter explosion for fully connected vs convolutional
print(f"{'Image size':>12} | {'FC first layer params':>22} | {'Conv 3x3 (32 filters) params':>30}")
print("-" * 72)
for H in [28, 32, 64, 224]:
    n_pixel = H * H * 3
    fc_params = n_pixel * 1000           # FC to 1000 units
    conv_params = 3 * 3 * 3 * 32 + 32   # 32 filters, 3×3×3 each + bias — SAME regardless of H
    print(f"{H:>5}×{H}×3 | {fc_params:>22,} | {conv_params:>30,}")

print()
print("The conv layer has the SAME number of parameters for any image size!")
print("This is weight sharing: the filter reuses the same weights everywhere.")


---
## Section 2 — Convolution from scratch

$$\text{Output}(i,j) = \sum_m \sum_n \text{Input}(i+m,\, j+n) \times \text{Filter}(m,n) + b$$

The filter slides over the input one position at a time (stride=1 by default).


In [ ]:
def convolve2d(inp, kernel, stride=1, padding=0):
    """2-D convolution (single channel, no bias)."""
    if padding > 0:
        inp = np.pad(inp, padding, mode='constant')
    Hi, Wi = inp.shape
    Hk, Wk = kernel.shape
    Ho = (Hi - Hk) // stride + 1
    Wo = (Wi - Wk) // stride + 1
    out = np.zeros((Ho, Wo))
    for i in range(Ho):
        for j in range(Wo):
            out[i, j] = (inp[i*stride:i*stride+Hk, j*stride:j*stride+Wk] * kernel).sum()
    return out

# Sobel edge detector filters (classic hand-engineered features)
sobel_x = np.array([[ 1, 0,-1],[ 2, 0,-2],[ 1, 0,-1]], dtype=float)  # vertical edges
sobel_y = np.array([[ 1, 2, 1],[ 0, 0, 0],[-1,-2,-1]], dtype=float)  # horizontal edges

# Toy input: a bright square on a dark background
inp = np.zeros((8, 8))
inp[2:6, 2:6] = 1.0

edge_x = convolve2d(inp, sobel_x)
edge_y = convolve2d(inp, sobel_y)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, data, title, cmap in zip(axes, [inp, edge_x, edge_y],
    ["Input (8×8)", "Sobel-X (vertical edges)", "Sobel-Y (horizontal edges)"],
    ["Greys_r", "RdBu", "RdBu"]):
    ax.imshow(data, cmap=cmap); ax.set_title(title, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print(f"Input shape: {inp.shape} → Sobel-X output: {edge_x.shape}  (6×6, shrank by 2)")


In [ ]:
# Verify the output size formula
def conv_out_size(W, F, P, S):
    """W_out = floor((W_in - F + 2P) / S) + 1"""
    return (W - F + 2*P) // S + 1

# Slide examples
print("Output size checks (from lecture slides):")
tests = [
    (7,  3, 1, 1, "7×7 + pad 1 + F=3"),
    (32, 5, 2, 1, "32×32 + pad 2 + F=5"),
    (227,11, 0, 4, "AlexNet CONV1"),
    (32, 3, 0, 2, "stride 2"),
]
for W, F, P, S, note in tests:
    out = conv_out_size(W, F, P, S)
    print(f"  W={W}, F={F}, P={P}, S={S}  →  {out}×{out}   [{note}]")


In [ ]:
# Practice 2.1 — complete the output size function and verify
def my_conv_out(W_in, F, P, S):
    # YOUR CODE HERE
    pass

# Verify against the slide examples:
print(my_conv_out(7,  3, 1, 1), "should be 7")
print(my_conv_out(227,11, 0, 4), "should be 55")
print(my_conv_out(32, 5, 2, 1), "should be 32")


---
## Section 3 — Sparse connections and weight sharing

**Sparse connections:** each output neuron only connects to a small local region (the receptive field), not all inputs.

**Weight sharing:** the SAME filter weights are reused at every position — only 3 unique weights for a filter of width 3, regardless of input size.

Together these give: fewer parameters + position-invariant features.


In [ ]:
# Demonstrate weight sharing: the same filter applied at multiple positions
inp_1d = np.array([1.0, 3.0, 2.0, 5.0, 1.0, 4.0, 2.0])  # 1D input
filt   = np.array([1.0, 2.0, 1.0])                        # 1D filter

# Apply at positions 0..4 (stride=1, no padding)
outputs = []
for start in range(len(inp_1d) - len(filt) + 1):
    patch = inp_1d[start : start + len(filt)]
    out   = (patch * filt).sum()
    outputs.append(out)
    print(f"Position {start}: patch={patch}  dot  filt={filt} = {out:.1f}")

print(f"Weights: {filt}  (only 3 weights shared across all positions!)")
print(f"FC would need {len(inp_1d) * len(outputs)} weights for the same mapping")


In [ ]:
# Parameter count: FC vs Conv
print("Parameter count comparison — same mapping:")
W_in = 100   # input size
W_out = 98   # output size (no padding, F=3)

fc_params   = W_in * W_out          # fully connected
conv_params = 3                      # just the filter weights (+ bias)

print(f"  Fully connected: {W_in} × {W_out} = {fc_params} parameters")
print(f"  Convolution (F=3): {conv_params} parameters")
print(f"  Ratio: {fc_params / conv_params:.0f}x more parameters for FC!")


---
## Section 4 — Stride and padding

**Stride S** = how many pixels the filter moves each step.  
- S=1: dense output, slow  
- S=2: halves spatial size, faster

**Padding P** = zeros added around the border to prevent shrinkage.  
Standard "same" padding: `P = (F-1) / 2`


In [ ]:
# Compare stride=1 vs stride=2 on the same input
inp2 = np.random.rand(6, 6)
filt2 = np.ones((3,3)) / 9   # averaging filter

out_s1 = convolve2d(inp2, filt2, stride=1, padding=0)
out_s2 = convolve2d(inp2, filt2, stride=2, padding=0)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, data, title in zip(axes,
    [inp2, out_s1, out_s2],
    ["Input (6×6)", f"stride=1 → {out_s1.shape[0]}×{out_s1.shape[0]}", f"stride=2 → {out_s2.shape[0]}×{out_s2.shape[0]}"]):
    ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print("Formula: W_out = floor((W_in - F + 2P) / S) + 1")
for S in [1, 2]:
    out = (6 - 3 + 0) // S + 1
    print(f"  S={S}: ({6}-{3}+0)/{S} + 1 = {out}")


In [ ]:
# Padding: demonstrate 'same' padding preserves size
def convolve2d_pad(inp, kernel, stride=1, padding=0):
    return convolve2d(inp, kernel, stride=stride, padding=padding)

inp3 = np.random.rand(5, 5)
filt3 = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=float)  # sharpening filter

# Without padding
out_no_pad = convolve2d_pad(inp3, filt3, padding=0)
# With padding P=1 (same padding for F=3)
out_pad    = convolve2d_pad(inp3, filt3, padding=1)

print(f"Input:           {inp3.shape}")
print(f"No padding:      {out_no_pad.shape}  ← shrank!")
print(f"Padding P=1:     {out_pad.shape}   ← same size preserved ✓")
print()
print("Same-padding rule: P = (F-1)/2")
for F in [3, 5, 7]:
    print(f"  F={F}: P = ({F}-1)/2 = {(F-1)//2}")


---
## Section 5 — Pooling

Pooling **reduces spatial size** (typically by 2×) and builds **translation invariance**.  
It has **zero learnable parameters** — it is a fixed downsampling operation.

Most common: **Max pooling** (takes the max in each window).


In [ ]:
def max_pool2d(x, size=2, stride=2):
    H, W = x.shape
    Ho = (H - size) // stride + 1
    Wo = (W - size) // stride + 1
    out = np.zeros((Ho, Wo))
    for i in range(Ho):
        for j in range(Wo):
            out[i,j] = x[i*stride:i*stride+size, j*stride:j*stride+size].max()
    return out

def avg_pool2d(x, size=2, stride=2):
    H, W = x.shape
    Ho = (H - size) // stride + 1
    Wo = (W - size) // stride + 1
    out = np.zeros((Ho, Wo))
    for i in range(Ho):
        for j in range(Wo):
            out[i,j] = x[i*stride:i*stride+size, j*stride:j*stride+size].mean()
    return out

# Demo from lecture slides
inp_pool = np.array([[2.,5.,1.,3.],[9.,4.,7.,2.],[1.,6.,3.,8.],[4.,2.,5.,1.]])
mp = max_pool2d(inp_pool)
ap = avg_pool2d(inp_pool)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, data, title in zip(axes, [inp_pool, mp, ap],
    ["Input (4×4)", "Max pool (2×2)", "Avg pool (2×2)"]):
    ax.imshow(data, cmap="Blues", vmin=0, vmax=9)
    ax.set_title(title, fontweight="bold"); ax.set_xticks([]); ax.set_yticks([])
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, str(int(data[i,j])), ha="center", va="center",
                    fontsize=13, fontweight="bold",
                    color="white" if data[i,j]>5 else "navy")
plt.tight_layout(); plt.show()
print(f"4×4 → max pool 2×2 → {mp.shape}  (spatial size halved)")


In [ ]:
# Translation invariance demo from lecture
sig = np.zeros((4, 4)); sig[1,1] = 1.0     # signal at (1,1)
sig2 = np.zeros((4, 4)); sig2[1,2] = 1.0   # signal shifted one pixel right

pool1 = max_pool2d(sig)
pool2 = max_pool2d(sig2)

print("Signal at (1,1):")
print(pool1)
print("\nSignal shifted to (1,2):")
print(pool2)
print("\nOutputs identical?", np.array_equal(pool1, pool2))
print("→ Max pooling provides invariance to small translations!")


In [ ]:
# Practice 5.1 — trace the spatial dimensions through multiple pool layers
W = 32   # starting size

print(f"Input: {W}×{W}")
for i in range(1, 4):
    W = (W - 2) // 2 + 1   # pool size=2, stride=2, no padding
    print(f"After pool {i}: {W}×{W}")

print()
print("General rule: each 2×2 pool with stride 2 halves the spatial dimension")
print("Two pools: 32×32 → 16×16 → 8×8")


---
## Section 6 — Activation function: ReLU

$$\text{ReLU}(z) = \max(0, z)$$

$$\text{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \le 0 \end{cases}$$

Why ReLU dominates in deep learning:
- **No vanishing gradient** for positive inputs (gradient is always 1 or 0)
- **Computationally trivial** (just a threshold comparison)
- **Sparse activation** — many neurons output exactly 0


In [ ]:
z = np.linspace(-4, 4, 400)
relu     = np.maximum(0, z)
relu_d   = np.where(z > 0, 1.0, 0.0)
leaky    = np.where(z >= 0, z, 0.1 * z)
sigmoid  = 1 / (1 + np.exp(-z))
sigmoid_d = sigmoid * (1 - sigmoid)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(z, relu,    color="#C0392B", lw=2.5, label="ReLU")
axes[0].plot(z, leaky,   color="#1ABC9C", lw=2, ls="--", label="Leaky ReLU (α=0.1)")
axes[0].plot(z, sigmoid, color="#8E44AD", lw=1.5, ls=":", label="Sigmoid")
axes[0].axhline(0, color="grey", lw=0.5); axes[0].axvline(0, color="grey", lw=0.5)
axes[0].set_title("Activation functions"); axes[0].legend(frameon=False, fontsize=9)
axes[0].grid(alpha=0.2)

axes[1].plot(z, relu_d,   color="#C0392B", lw=2.5, label="ReLU gradient")
axes[1].plot(z, sigmoid_d,color="#8E44AD", lw=2, ls=":", label="Sigmoid gradient")
axes[1].axhline(0, color="grey", lw=0.5); axes[1].axvline(0, color="grey", lw=0.5)
axes[1].set_title("Gradients (backprop)"); axes[1].legend(frameon=False, fontsize=9)
axes[1].grid(alpha=0.2)
axes[1].set_ylim(-0.1, 1.3)

# Sparsity demo
np.random.seed(3)
fm = np.random.randn(5,5) * 2
fm_relu = np.maximum(0, fm)
axes[2].imshow(fm_relu, cmap="Blues", vmin=0)
axes[2].set_title(f"After ReLU: {100*(fm>0).mean():.0f}% active")
axes[2].set_xticks([]); axes[2].set_yticks([])
for i in range(5):
    for j in range(5):
        axes[2].text(j, i, f"{fm_relu[i,j]:.1f}", ha="center", va="center",
                     fontsize=9, color="white" if fm_relu[i,j]>1 else "navy")
plt.tight_layout(); plt.show()

print("Sigmoid gradient saturates near 0 for large |z| — vanishing gradient problem")
print("ReLU gradient is always 0 or 1 — no saturation for positive inputs")


In [ ]:
# Practice 6.1 — implement and compare activation functions
z_test = np.array([-3.2, -1.0, 0.0, 0.5, 2.1, 4.8])

# (a) ReLU
relu_z = # YOUR CODE HERE

# (b) Leaky ReLU (alpha=0.01)
leaky_z = # YOUR CODE HERE

# (c) What fraction are 'dead' (output exactly 0) after ReLU?
frac_dead = # YOUR CODE HERE

print("Input:     ", z_test)
print("ReLU:      ", relu_z)
print("Leaky ReLU:", leaky_z)
print(f"Dead neurons: {frac_dead:.1%}")


---
## Section 7 — Dropout and weight initialisation

**Dropout** randomly zeroes neurons during training, forcing the network to not rely on any single neuron. Acts as regularisation.

**He initialisation** sets weights so activation variance ≈ 1 throughout the network:
$$w \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n}}\right)$$
where $n$ = number of input connections.


In [ ]:
# Demonstrate: why weight initialisation matters
def forward_pass(X, W_list, activation):
    """Pass X through a deep network and track variance of each layer's output."""
    h = X
    variances = [X.var()]
    for W in W_list:
        h = activation(h @ W)
        variances.append(h.var())
    return variances

np.random.seed(0)
N, D = 100, 256   # batch size, layer width
depth = 10         # number of layers

X = np.random.randn(N, D)

# Three initialisation schemes
W_too_small = [np.random.randn(D,D) * 0.01 for _ in range(depth)]
W_too_large = [np.random.randn(D,D) * 1.0  for _ in range(depth)]
W_he        = [np.random.randn(D,D) * np.sqrt(2/D) for _ in range(depth)]

relu_fn = lambda x: np.maximum(0, x)

fig, ax = plt.subplots(figsize=(10, 4))
for W_list, lbl, col in [(W_too_small,"Too small (×0.01)","#C0392B"),
                          (W_too_large,"Too large (×1.0)","#E67E22"),
                          (W_he,      "He init (√2/n)","#27AE60")]:
    variances = forward_pass(X, W_list, relu_fn)
    ax.semilogy(variances, lw=2, color=col, label=lbl)

ax.set_xlabel("layer"); ax.set_ylabel("activation variance (log scale)")
ax.set_title("Weight initialisation: He init keeps variance stable across layers")
ax.legend(frameon=False); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print("Too small: signal shrinks → gradients vanish → network can't learn")
print("Too large: signal grows → activations explode → NaN/Inf")
print("He init:   variance stays roughly constant → stable training")


In [ ]:
# Dropout simulation
def dropout(X, p=0.5, training=True):
    """Apply dropout with keep-probability (1-p)."""
    if not training:
        return X * (1 - p)    # scale down at test time
    mask = (np.random.rand(*X.shape) > p).astype(float)
    return X * mask / (1 - p)  # scale up during training (inverted dropout)

np.random.seed(7)
activations = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])

print("Original activations:", activations)
print()
for trial in range(3):
    dropped = dropout(activations, p=0.5)
    n_zero = (dropped == 0).sum()
    print(f"Trial {trial+1}: {dropped.round(2)}   ({n_zero} neurons dropped)")
print()
print("Test time (no dropout, scale down):", dropout(activations, p=0.5, training=False))


---
## Section 8 — Batch normalisation

Batch norm normalises activations across the mini-batch, solving Internal Covariate Shift.

$$\hat{z} = \frac{z - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}$$
$$y = \gamma \hat{z} + \beta$$

where $\mu_B, \sigma_B^2$ are computed over the mini-batch, and $\gamma, \beta$ are learned.


In [ ]:
def batch_norm(Z, gamma=1.0, beta=0.0, eps=1e-8):
    """Batch normalisation for a batch of scalars."""
    mu    = Z.mean()
    sigma = Z.std()
    Z_hat = (Z - mu) / (sigma + eps)
    return gamma * Z_hat + beta, mu, sigma

# Example from lecture: people's weights + heights split by gender
np.random.seed(1)
weights_women  = np.random.normal(55,  8, 50)   # kg
weights_men    = np.random.normal(75, 10, 50)   # kg

print("Without batch norm — women batch then men batch:")
print(f"  Women: mean={weights_women.mean():.1f}, std={weights_women.std():.1f}")
print(f"  Men:   mean={weights_men.mean():.1f}, std={weights_men.std():.1f}")
print(f"  Shift between batches: {weights_men.mean() - weights_women.mean():.1f} kg!")
print()

norm_w, mu_w, sig_w = batch_norm(weights_women)
norm_m, mu_m, sig_m = batch_norm(weights_men)
print("After batch norm — both batches are now centred and scaled:")
print(f"  Women: mean={norm_w.mean():.4f}, std={norm_w.std():.4f}")
print(f"  Men:   mean={norm_m.mean():.4f}, std={norm_m.std():.4f}")
print("  The network sees consistently normalised inputs regardless of batch composition!")


In [ ]:
# Visualise the effect of batch norm on a deep network training
np.random.seed(0)
n_steps = 60

# Without batch norm — convergence is slow/unstable
loss_no_bn = np.exp(-np.arange(n_steps)/40) * 2 + 0.25 + 0.12 * np.random.randn(n_steps)
loss_no_bn = np.maximum(loss_no_bn, 0.2)

# With batch norm — faster, smoother convergence
loss_bn = np.exp(-np.arange(n_steps)/18) * 1.8 + 0.08 + 0.03 * np.random.randn(n_steps)
loss_bn = np.maximum(loss_bn, 0.07)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss_no_bn, color="#C0392B", lw=2, label="Without batch norm", alpha=0.8)
ax.plot(loss_bn,    color="#27AE60", lw=2, label="With batch norm")
ax.set_xlabel("training step"); ax.set_ylabel("loss")
ax.set_title("Batch normalisation: faster convergence, smoother training")
ax.legend(frameon=False); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()


In [ ]:
# Practice 8.1 — implement batch norm from scratch
def my_batch_norm(Z, gamma, beta, eps=1e-8):
    """
    Z:     (N,) array of activations over the mini-batch
    gamma: learned scale
    beta:  learned shift
    Returns normalised activations y and intermediate values (mu, sigma)
    """
    # YOUR CODE HERE
    mu    = ...
    sigma = ...
    Z_hat = ...
    y     = ...
    return y, mu, sigma

# Test
Z = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y, mu, sig = my_batch_norm(Z, gamma=2.0, beta=1.0)
print(f"Input:  {Z}")
print(f"Mean: {mu:.2f}  Std: {sig:.4f}")
print(f"Output: {y.round(4)}")
print(f"Check: output mean ≈ beta={1.0}? {abs(y.mean()-1.0)<0.01}")
print(f"Check: output std ≈ gamma={2.0}? {abs(y.std()-2.0)<0.01}")


---
## Section 9 — Build and train a CNN in PyTorch

We build a CNN for CIFAR-10 (10 classes: airplane, car, bird, cat, deer, dog, frog, horse, ship, truck).

Architecture follows the slide's typical structure: `[CONV-ReLU-POOL] × 2 → FC → Softmax`


In [ ]:
if not HAVE_TORCH:
    print("Install PyTorch to run sections 9-10:  pip install torch torchvision")
else:
    class CIFAR10_CNN(nn.Module):
        """Simple CNN for CIFAR-10 following the VE445 lecture structure."""
        def __init__(self):
            super().__init__()
            # Block 1: CONV → BatchNorm → ReLU → Pool
            self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 32×32×3 → 32×32×32
            self.bn1   = nn.BatchNorm2d(32)
            self.pool  = nn.MaxPool2d(2, 2)                          # halves spatial size

            # Block 2: CONV → BatchNorm → ReLU → Pool
            self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # 16×16×32 → 16×16×64

            self.bn2   = nn.BatchNorm2d(64)

            # Classification head
            self.dropout = nn.Dropout(p=0.5)
            self.fc1  = nn.Linear(64 * 8 * 8, 256)                  # 8×8×64 → 256
            self.fc2  = nn.Linear(256, 10)                           # 256 → 10 classes

        def forward(self, x):
            # Block 1
            x = self.pool(F.relu(self.bn1(self.conv1(x))))   # → (B, 32, 16, 16)
            # Block 2
            x = self.pool(F.relu(self.bn2(self.conv2(x))))   # → (B, 64, 8, 8)
            # Flatten
            x = x.view(x.size(0), -1)                         # → (B, 4096)
            x = self.dropout(F.relu(self.fc1(x)))             # → (B, 256)
            x = self.fc2(x)                                    # → (B, 10)
            return x

    model = CIFAR10_CNN().to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {n_params:,}")
    print()
    # Verify forward pass shapes
    dummy = torch.zeros(4, 3, 32, 32).to(DEVICE)
    with torch.no_grad():
        out = model(dummy)
    print(f"Input:  {tuple(dummy.shape)}")
    print(f"Output: {tuple(out.shape)}  (4 images × 10 class logits)")


In [ ]:
if HAVE_TORCH:
    # Download CIFAR-10 and train
    print("Downloading CIFAR-10...")
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),             # data augmentation
        transforms.RandomCrop(32, padding=4),          # data augmentation
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),   # CIFAR-10 mean
                             (0.2023,0.1994,0.2010)),   # CIFAR-10 std
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
    ])

    train_ds = datasets.CIFAR10("/tmp/cifar10", train=True,  download=True, transform=transform_train)
    test_ds  = datasets.CIFAR10("/tmp/cifar10", train=False, download=True, transform=transform_test)
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
    test_dl  = torch.utils.data.DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)
    print(f"Train: {len(train_ds):,}   Test: {len(test_ds):,}")


In [ ]:
if HAVE_TORCH:
    EPOCHS = 10
    opt   = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.1)
    loss_fn = nn.CrossEntropyLoss()

    CLASSES = ["plane","car","bird","cat","deer","dog","frog","horse","ship","truck"]
    history = {"train_loss": [], "test_acc": []}

    for ep in range(1, EPOCHS+1):
        model.train()
        losses = []
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(imgs), labels)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        sched.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in test_dl:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total += len(labels)
        acc = correct / total
        history["train_loss"].append(np.mean(losses))
        history["test_acc"].append(acc)
        print(f"Epoch {ep:>2}/{EPOCHS}  loss={history['train_loss'][-1]:.4f}  test_acc={acc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history["train_loss"], "o-", color="#1F4E79", lw=2)
    axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch"); axes[0].grid(alpha=0.25)
    axes[1].plot([a*100 for a in history["test_acc"]], "s-", color="#27AE60", lw=2)
    axes[1].set_title("Test accuracy (%)"); axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.25)
    plt.tight_layout(); plt.show()


---
## Section 10 — Famous architectures

### AlexNet (2012) — the breakthrough
- **16.4% top-5 error** (previous best ~26%)
- First use of ReLU, Dropout, data augmentation at scale
- 5 conv layers + 3 FC layers, trained on 2 GPUs for 5-6 days

### VGGNet (2014) — simplicity wins
- **7.3% top-5 error** with only 3×3 filters
- Key insight: two 3×3 layers ≡ one 5×5 receptive field, fewer params, extra non-linearity

### GoogLeNet (2014) — efficiency
- **6.7% top-5 error** with only **5M parameters** (12× less than AlexNet!)
- Inception modules: parallel 1×1, 3×3, 5×5 branches concatenated
- 1×1 conv for channel reduction before expensive operations

### ResNet (2015) — go deep
- **3.6% top-5 error** (better than human at ~5%)
- Skip connections: `output = F(x) + x`
- Enables training of 152-layer networks without degradation


In [ ]:
# Compute parameter counts for the famous architectures
def conv_params(F, Cin, Cout): return F*F*Cin*Cout + Cout
def fc_params(Nin, Nout):      return Nin*Nout + Nout

# AlexNet CONV1 worked example from the slides
p_alex1 = conv_params(11, 3, 96)
print(f"AlexNet CONV1 (11×11, 3→96): {p_alex1:,} ≈ 35K  (matches slide!)")

# VGGNet: why 3×3 is better than larger filters
print()
print("VGGNet: two 3×3 layers vs one 5×5 layer (same receptive field):")
two_3x3 = 2 * conv_params(3, 64, 64)
one_5x5 = conv_params(5, 64, 64)
print(f"  Two 3×3:  {two_3x3:,} params")
print(f"  One 5×5:  {one_5x5:,} params")
print(f"  Saving: {100*(one_5x5-two_3x3)/one_5x5:.0f}%  (plus one extra ReLU)")

# GoogLeNet: 1×1 reduces computation before 3×3
print()
print("GoogLeNet inception module (simplified):")
print("  Without 1×1: 1×1×192×128 + 3×3×192×256 = ", conv_params(1,192,128) + conv_params(3,192,256))
print("  With 1×1 first: 1×1×192×32 + 3×3×32×256 = ", conv_params(1,192,32) + conv_params(3,32,256))


In [ ]:
# ResNet skip connection — why it works
if HAVE_TORCH:
    class ResBlock(nn.Module):
        """Basic ResNet residual block."""
        def __init__(self, channels):
            super().__init__()
            self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
            self.bn1   = nn.BatchNorm2d(channels)
            self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
            self.bn2   = nn.BatchNorm2d(channels)

        def forward(self, x):
            residual = x                           # the skip connection
            out = F.relu(self.bn1(self.conv1(x)))
            out = self.bn2(self.conv2(out))
            out = out + residual                   # add the original input back
            return F.relu(out)

    block = ResBlock(64)
    x = torch.randn(1, 64, 16, 16)
    y = block(x)
    print(f"ResBlock: input {tuple(x.shape)} → output {tuple(y.shape)}")
    print()
    print("If conv weights are zero: F(x) = 0, so output = 0 + x = x")
    print("This means the block defaults to an identity function,")
    print("which is why very deep ResNets still train: later layers can simply")
    print("pass the input through unchanged if they don't need to change it.")


In [ ]:
# Practice 10.1 — parameter counting for VGG-style network
# Compute total parameters for this VGG-like architecture:
# Input: 32×32×3
# CONV(3×3, 3→64) → ReLU → CONV(3×3, 64→64) → ReLU → MaxPool(2×2)
# CONV(3×3, 64→128) → ReLU → CONV(3×3, 128→128) → ReLU → MaxPool(2×2)
# FC(8×8×128 → 512) → ReLU → FC(512 → 10)

# YOUR CODE HERE
params = {
    "conv1": conv_params(3, 3, 64),
    "conv2": conv_params(3, 64, 64),
    "conv3": # YOUR CODE HERE,
    "conv4": # YOUR CODE HERE,
    "fc1":   # YOUR CODE HERE,
    "fc2":   # YOUR CODE HERE,
}

for name, p in params.items():
    print(f"{name:<6}: {p:>10,}")
print(f"{'Total':<6}: {sum(params.values()):>10,}")


---
## Appendix — Architecture summary table

| Network | Year | Layers | Params | ILSVRC Top-5 | Key idea |
|---|---|---|---|---|---|
| LeNet-5 | 1998 | 7 | ~60K | — | First CNN |
| AlexNet | 2012 | 8 | 60M | 16.4% | ReLU, Dropout, GPU |
| VGGNet | 2014 | 16/19 | 138M | 7.3% | All 3×3 filters |
| GoogLeNet | 2014 | 22 | 5M | 6.7% | Inception, 1×1 conv |
| ResNet-50 | 2015 | 50 | 25M | 5.3% | Skip connections |
| ResNet-152 | 2015 | 152 | 60M | 3.6% | Very deep residuals |
| DenseNet | 2017 | 121 | 8M | — | Dense skip connections |

> The key trend: **architectural innovations matter more than just making networks bigger.**  
> GoogLeNet outperforms AlexNet with 12× fewer parameters.  
> ResNet enables depths of 150+ layers through a simple identity shortcut.
